In [1]:
import numpy as np
import pandas as pd
from glob import glob
from matplotlib import pyplot as plt
import seaborn as sns
import pickle as pkl
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from one.api import ONE

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
one = ONE()

In [5]:
units_df = bwm_units(one)

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01


In [6]:
neuron_counts = units_df.groupby(["Beryl", "eid"]).size().reset_index(name="neuron_count")

In [7]:
valid_pairings = neuron_counts[neuron_counts["neuron_count"] >= 5]

In [8]:
valid_pairings  # atleast 10 neurons

,Beryl,eid,neuron_count
0,AAA,37e96d0b-5b4b-4c6e-9b29-7edbdc94bbd0,13
1,AAA,57b5ae8f-d446-4161-b439-b191c5e3e77b,9
2,ACAd,0f77ca5d-73c2-45bd-aa4c-4c5ed275dbde,19
3,ACAd,113c5b6c-940e-4b21-b462-789b4c2be0e5,19
4,ACAd,196a2adf-ff83-49b2-823a-33f990049c2e,30
...,...,...,...
2473,ZI,f8d5c8b0-b931-4151-b86c-c471e2e80e5d,7
2474,x,004d8fd5-41e7-4f1b-a45b-0d4ad76fe446,10
2475,x,20c112a1-8a42-44e0-a4cd-e5b932f7bda9,11
2476,x,549caacc-3bd7-40f1-913d-e94141816547,15


In [9]:
final_counts = (
    valid_pairings.groupby("Beryl")["eid"]
    .nunique()
    .reset_index(name="valid_eid_count")
    .sort_values(by="valid_eid_count", ascending=False)
)

In [10]:
final_counts  # now we look at only regions with more than 20 sessions

,Beryl,valid_eid_count
97,MRN,99
22,CA1,94
41,DG,68
34,CP,64
80,LP,58
...,...,...
74,LDT,2
81,LPO,2
100,NB,2
82,LRN,2


In [11]:
regions_of_interest = final_counts[final_counts["valid_eid_count"] >= 20]["Beryl"].values

In [12]:
regions_of_interest

array(['MRN', 'CA1', 'DG', 'CP', 'LP', 'SCm', 'APN', 'CA3', 'PO', 'PAG',
       'MOs', 'VISp', 'VPM', 'VISa', 'MOp', 'ZI', 'LSr', 'IRN', 'SUB',
       'CUL4 5', 'IP', 'PIR', 'RSPv', 'LGd', 'SSp-bfd', 'RSPd', 'ACAd',
       'PRNr', 'MV', 'RT'], dtype=object)

In [13]:
region_totals = units_df.groupby("Beryl").size()
valid_regions = region_totals[region_totals >= 20].index
df_valid = units_df[units_df["Beryl"].isin(valid_regions)]

final_table = (
    df_valid.groupby("Beryl")["eid"]
    .agg(
        total_neurons="size",
        unique_eid_count="nunique",
        eids_list=lambda x: list(x.unique()),
    )
    .reset_index()
    .sort_values(by="total_neurons", ascending=False)
)

In [14]:
final_table[final_table["unique_eid_count"] >= 10]["Beryl"].values

array(['CP', 'MRN', 'PO', 'LP', 'CA1', 'SCm', 'APN', 'MOp', 'VPM', 'MOs',
       'CUL4 5', 'LSr', 'DG', 'VISa', 'SUB', 'SIM', 'CA3', 'VISp', 'PRNr',
       'IRN', 'PAG', 'MV', 'CENT3', 'ENTm', 'IP', 'CENT2', 'PIR', 'MD',
       'ENTl', 'LD', 'GRN', 'LGd', 'ANcr2', 'RSPv', 'SSp-bfd', 'IC',
       'AON', 'SSs', 'ANcr1', 'VPL', 'RN', 'RT', 'PL', 'PARN', 'CS',
       'PRM', 'ProS', 'VISam', 'ACAd', 'PPN', 'Eth', 'SSp-m', 'ACB', 'MG',
       'SPIV', 'ZI', 'GPe', 'SPVI', 'DP', 'DCO', 'RSPd', 'RSPagl',
       'SSp-tr', 'PB', 'COPY', 'SUV', 'TTd', 'SI', 'VISpm', 'POST', 'PoT',
       'BST', 'ACAv', 'BMA', 'SCs', 'LHA', 'CEA', 'LSv', 'POL', 'BLA',
       'SNr', 'VM', 'NTS'], dtype=object)

In [18]:
final_table["eids_list"][1]

['0f77ca5d-73c2-45bd-aa4c-4c5ed275dbde',
 '113c5b6c-940e-4b21-b462-789b4c2be0e5',
 '196a2adf-ff83-49b2-823a-33f990049c2e',
 '259927fd-7563-4b03-bc5d-17b4d0fa7a55',
 '28741f91-c837-4147-939e-918d38d849f2',
 '5455a21c-1be7-4cae-ae8e-8853a8d5f55e',
 '5ec72172-3901-4771-8777-6e9490ca51fc',
 '6f6d2c8e-28be-49f4-ae4d-06be2d3148c1',
 '78b4fff5-c5ec-44d9-b5f9-d59493063f00',
 '9468fa93-21ae-4984-955c-e8402e280c83',
 '9e9c6fc0-4769-4d83-9ea4-b59a1230510e',
 'a4000c2f-fa75-4b3e-8f06-a7cf599b87ad',
 'a92c4b1d-46bd-457e-a1f4-414265f0e2d4',
 'aad23144-0e52-4eac-80c5-c4ee2decb198',
 'cc45c568-c3b9-4f74-836e-c87762e898c8',
 'd2918f52-8280-43c0-924b-029b2317e62c',
 'd7e60cc3-6020-429e-a654-636c6cc677ea',
 'dfbe628d-365b-461c-a07f-8b9911ba83aa',
 'ebc9392c-1ecb-4b4b-a545-4e3d70d23611',
 'fb70ebf7-8175-42b0-9b7a-7c6e8612226e',
 'ff4187b5-4176-4e39-8894-53a24b7cf36b']